# WORKSHOP 2 - CLASIFICACIÓN: Detección de Fatiga Muscular en Ciclismo

## Objetivo General
Detectar si un ciclista está en condición normal o tiene desgaste muscular (fatiga) a partir de señales EMG (electromiografía) de 8 músculos de la pierna.

## Dataset
- **Nombre:** Muscle Fatigue Cycling
- **Fuente:** HuggingFace - YominE/Muscle_Fatigue_Cycling
- **Características:** 8 señales de músculos, frecuencia de muestreo 1000 Hz
- **Target:** Estado muscular (0=normal, 1=fatiga, 2=fatiga convertido a 1)

## Estructura del Trabajo
1. Cargar datos y exploración preliminar
2. Extracción de características (feature engineering)
3. EDA completo
4. Preprocesamiento y normalización
5. Entrenamiento de 5 modelos
6. Evaluación y comparación
7. Prueba con datos artificiales

## PARTE 1: Importar librerías y cargar datos

In [22]:
# Importar todas las librerías necesarias
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.signal import welch
import warnings
warnings.filterwarnings('ignore')

# Machine Learning
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

# Deep Learning
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

# Cargar dataset
from datasets import load_dataset

print("✓ Librerías importadas correctamente")
print("✓ Iniciando carga del dataset...")

# Cargar el dataset desde HuggingFace
dataset = load_dataset("YominE/Muscle_Fatigue_Cycling")
train_data = dataset['train']

print(f"✓ Dataset cargado: {len(train_data)} muestras")
print(f"✓ Columnas disponibles: {train_data.column_names}")

✓ Librerías importadas correctamente
✓ Iniciando carga del dataset...
✓ Dataset cargado: 3002137 muestras
✓ Columnas disponibles: ['Time', 'Right Rectus femoris', 'Left Gluteus maximus', 'Left Gastrocnemius medialis', 'Left Semitendinosus', 'Left Biceps femoris caput longus', 'Right Vastus medialis', 'Right Tibialis anterior', 'Left Gastrocnemius lateralis', 'Target']


In [ ]:
# Convertir a DataFrame para facilitar el manejo
# Para evitar cargar todo a la memoria, usamos un subconjunto representativo
# Usaremos las primeras 100,000 muestras (100 segundos de datos)

n_samples = min(100000, len(train_data))
print(f"Cargando {n_samples} muestras para procesamiento...")

# Método correcto: usar to_pandas() o crear DataFrame directamente
# Obtenemos los nombres de columnas del dataset
column_names = train_data.column_names
print(f"Columnas del dataset: {column_names}")

# Extraer datos en lotes para eficiencia
datos_dict = {col: [] for col in column_names}

for i in range(0, n_samples, 10000):
    batch = train_data[i:min(i+10000, n_samples)]
    for col in column_names:
        datos_dict[col].extend(batch[col])

# Crear DataFrame desde diccionario (preserva nombres de columnas)
df = pd.DataFrame(datos_dict)

print(f"✓ Datos cargados en DataFrame: {df.shape}")
print(f"\nColumnas del DataFrame: {df.columns.tolist()}")
print(f"\nPrimeras filas:")
print(df.head())

print(f"\nInfo del dataset:")
print(f"Total de muestras: {len(df)}")
print(f"Tipos de datos:\n{df.dtypes}")

Cargando 100000 muestras para procesamiento...
✓ Datos cargados en DataFrame: (100, 1)

Primeras filas:
                             0
0                         Time
1         Right Rectus femoris
2         Left Gluteus maximus
3  Left Gastrocnemius medialis
4          Left Semitendinosus

Info del dataset:
Total de muestras: 100
Columnas: [0]
Tipos de datos: {0: dtype('O')}


In [24]:
# Preprocesar Target: convertir a problema binario
# Convertir etiqueta 2 por 1 como indica el trabajo

print("\n" + "="*70)
print("PREPROCESAMIENTO DEL TARGET")
print("="*70)

print(f"\nValores únicos del target ANTES: {sorted(df['Target'].unique())}")
print(f"Distribución ANTES:")
print(df['Target'].value_counts().sort_index())

# Convertir 2 → 1 (ambas representan fatiga)
df['Target'] = df['Target'].replace(2, 1)

print(f"\n✓ Target convertido a binario")
print(f"Valores únicos del target DESPUÉS: {sorted(df['Target'].unique())}")
print(f"\nDistribución DESPUÉS:")
print(df['Target'].value_counts().sort_index())

print("\n" + "-"*70)
print("INTERPRETACIÓN:")
print("-"*70)
print("0 = Condición muscular NORMAL (sin fatiga)")
print("1 = Desgaste MUSCULAR o FATIGA")
print(f"\nBalance de clases:")
print(f"  - {sum(df['Target']==0)} muestras en condición normal")
print(f"  - {sum(df['Target']==1)} muestras con fatiga")
print(f"  - Proporción: {sum(df['Target']==1)/len(df)*100:.1f}% fatigados")


PREPROCESAMIENTO DEL TARGET


KeyError: 'Target'

## PARTE 2: Extracción de Características (Feature Engineering)

**¿Por qué extracción de características?**
- Las señales crudas tienen 1000 muestras por segundo (muy voluminosas)
- La fatiga ocurre a nivel de SEGUNDOS, no de milisegundos individuales
- Extraeremos 7 características por músculo por ventana de 1 segundo
- Total: 8 músculos × 7 características = 56 características de entrada

In [ ]:
def extraer_caracteristicas(ventana, fs=1000):
    """
    Extrae 7 características de tiempo y frecuencia de una ventana de 1 segundo.
    
    DOMINIO DEL TIEMPO:
    - RMS: energía de la señal = RAÍZ(media(señal²))
    - Varianza: dispersión de los valores = media((señal - media)²)
    - ZCR: Zero Crossing Rate = cuántas veces cruza por cero
    - MAV: Mean Absolute Value = media(|señal|)
    
    DOMINIO DE LA FRECUENCIA:
    - Potencia Total: suma de potencias en todas las frecuencias
    - Frecuencia Media: promedio ponderado de frecuencias
    - Frecuencia Mediana (MDF): frecuencia que divide potencia en 50-50
    """
    
    # DOMINIO DEL TIEMPO
    rms = np.sqrt(np.mean(ventana**2))
    varianza = np.var(ventana)
    zcr = np.sum(np.diff(np.sign(ventana)) != 0)
    mav = np.mean(np.abs(ventana))
    
    # DOMINIO DE LA FRECUENCIA (Welch)
    freqs, psd = welch(ventana, fs=fs, nperseg=min(256, len(ventana)))
    
    potencia_total = np.sum(psd)
    
    # Frecuencia media (promedio ponderado)
    if np.sum(psd) > 0:
        frecuencia_media = np.sum(freqs * psd) / np.sum(psd)
    else:
        frecuencia_media = 0
    
    # Frecuencia mediana (MDF)
    potencia_acum = np.cumsum(psd)
    potencia_mitad = potencia_acum[-1] / 2
    idx_mediana = np.searchsorted(potencia_acum, potencia_mitad)
    frecuencia_mediana = freqs[idx_mediana] if idx_mediana < len(freqs) else 0
    
    return [rms, varianza, zcr, mav, potencia_total, frecuencia_media, frecuencia_mediana]


# Definir parámetros de ventanas
fs = 1000  # Frecuencia de muestreo en Hz
ventana_size = fs  # 1000 muestras = 1 segundo

# Nombres de los 8 músculos en el dataset
musculos = [col for col in df.columns if col not in ['Time', 'Target']]
musculos = sorted(musculos)  # Para consistencia

print(f"\n{'='*70}")
print("EXTRACCIÓN DE CARACTERÍSTICAS")
print(f"{'='*70}")
print(f"Tamaño de ventana: {ventana_size} muestras (1 segundo)")
print(f"Músculos a procesar: {len(musculos)}")
print(f"Características por músculo: 7")
print(f"Total características: {len(musculos) * 7}")

# Calcular número de ventanas completas
n_ventanas = len(df) // ventana_size

print(f"Total de muestras: {len(df)}")
print(f"Ventanas completas: {n_ventanas}")
print(f"\nProcesando ventanas...")

# Construir nuevo dataset con características
filas = []
nombres_feat = ['rms', 'var', 'zcr', 'mav', 'pot_total', 'freq_media', 'freq_mediana']

for idx_ventana in range(n_ventanas):
    inicio = idx_ventana * ventana_size
    fin = inicio + ventana_size
    
    ventana_df = df.iloc[inicio:fin]
    fila = {}
    
    # Para cada músculo, extraer características
    for musculo in musculos:
        ventana_senal = ventana_df[musculo].values
        caracteristicas = extraer_caracteristicas(ventana_senal, fs=fs)
        
        for nombre, valor in zip(nombres_feat, caracteristicas):
            fila[f'{musculo}_{nombre}'] = valor
    
    # El target de la ventana es el más común en esos 1000 samples
    fila['Target'] = ventana_df['Target'].mode()[0]
    
    filas.append(fila)
    
    if (idx_ventana + 1) % max(1, n_ventanas//10) == 0:
        print(f"  Procesadas {idx_ventana + 1}/{n_ventanas} ventanas...")

# Crear nuevo DataFrame con características
df_features = pd.DataFrame(filas)

print(f"\n✓ Dataset de características creado!")
print(f"Forma: {df_features.shape[0]} ventanas × {df_features.shape[1]} columnas")
print(f"\nPrimeras filas del nuevo dataset:")
print(df_features.head())

# Verificar balance
print(f"\nBalance de clases en el nuevo dataset:")
print(df_features['Target'].value_counts())

## PARTE 3: Análisis Exploratorio de Datos (EDA)

In [ ]:
print("\n" + "="*70)
print("ANÁLISIS EXPLORATORIO DE DATOS (EDA)")
print("="*70)

# Estadísticos básicos
print("\nEstadísticos descriptivos de las características:")
print(df_features.describe())

# Visualizar distribuciones de algunas características
feature_columns = [col for col in df_features.columns if col != 'Target']
sample_features = [col for col in feature_columns if 'rms' in col or 'freq_mediana' in col][:8]

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
fig.suptitle('Distribuciones de Características por Clase', fontsize=14, fontweight='bold')

for idx, feature in enumerate(sample_features):
    row = idx // 4
    col = idx % 4
    
    axes[row, col].hist(df_features[df_features['Target'] == 0][feature], 
                        bins=20, alpha=0.6, label='Normal', color='green')
    axes[row, col].hist(df_features[df_features['Target'] == 1][feature], 
                        bins=20, alpha=0.6, label='Fatiga', color='red')
    axes[row, col].set_title(feature, fontweight='bold')
    axes[row, col].set_ylabel('Frecuencia')
    axes[row, col].legend()
    axes[row, col].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n✓ Visualización 1: Distribuciones de características separadas por clase")

# Análisis de correlación
print("\nCalculando correlación con el target...")
correlaciones_target = df_features[feature_columns + ['Target']].corr()['Target'].drop('Target').abs()
top_features = correlaciones_target.nlargest(12)

print("\nTop 12 características más correlacionadas con Fatiga:")
print(top_features)

plt.figure(figsize=(10, 6))
top_features.plot(kind='barh', color='steelblue')
plt.title('Características Más Correlacionadas con Fatiga Muscular', fontweight='bold')
plt.xlabel('Correlación Absoluta')
plt.tight_layout()
plt.show()

print("\n✓ Visualización 2: Características más informativas")

# Boxplots para ver separabilidad
fig, axes = plt.subplots(2, 4, figsize=(16, 10))
fig.suptitle('Separabilidad de Clases - Boxplots', fontsize=14, fontweight='bold')

for idx, feature in enumerate(sample_features):
    row = idx // 4
    col = idx % 4
    
    data_normal = df_features[df_features['Target'] == 0][feature]
    data_fatiga = df_features[df_features['Target'] == 1][feature]
    
    axes[row, col].boxplot([data_normal, data_fatiga], labels=['Normal', 'Fatiga'])
    axes[row, col].set_title(feature, fontweight='bold')
    axes[row, col].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print("\n✓ Visualización 3: Separabilidad entre clases con boxplots")

## PARTE 4: Preprocesamiento de Datos

In [ ]:
print("\n" + "="*70)
print("PREPROCESAMIENTO Y NORMALIZACIÓN")
print("="*70)

# Preparar X (features) e y (target)
X = df_features[feature_columns].values
y = df_features['Target'].values

print(f"\nForma de X: {X.shape}")
print(f"Forma de y: {y.shape}")

# División 1: train+val (85%) vs test (15%)
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.15, random_state=42, stratify=y
)

# División 2: train (70%) vs val (15%)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.176, random_state=42, stratify=y_temp
)

print(f"\n✓ División de datos:")
print(f"  Train: {X_train.shape[0]} muestras ({X_train.shape[0]/len(X)*100:.1f}%)")
print(f"  Validación: {X_val.shape[0]} muestras ({X_val.shape[0]/len(X)*100:.1f}%)")
print(f"  Test: {X_test.shape[0]} muestras ({X_test.shape[0]/len(X)*100:.1f}%)")

# Normalización con StandardScaler (solo fit en train)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print(f"\n✓ Datos normalizados (0 media, 1 desv. estándar):")
print(f"  Media en train: {X_train_scaled.mean():.6f}")
print(f"  Desv. Est. en train: {X_train_scaled.std():.6f}")

# Verificar balance
print(f"\n✓ Balance de clases:")
print(f"  Train: {sum(y_train==0)} normales, {sum(y_train==1)} fatigados")
print(f"  Val: {sum(y_val==0)} normales, {sum(y_val==1)} fatigados")
print(f"  Test: {sum(y_test==0)} normales, {sum(y_test==1)} fatigados")

## PARTE 5: Entrenamiento y Comparación de 5 Modelos

In [ ]:
print("\n" + "="*70)
print("ENTRENAMIENTO DE 5 MODELOS DE CLASIFICACIÓN")
print("="*70)

modelos = {}
resultados = {}

# ===== MODELO 1: k-Nearest Neighbors =====
print("\n1. Entrenando k-Nearest Neighbors...")
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train_scaled, y_train)
modelos['kNN'] = knn

y_train_pred_knn = knn.predict(X_train_scaled)
y_val_pred_knn = knn.predict(X_val_scaled)
y_test_pred_knn = knn.predict(X_test_scaled)

resultados['kNN'] = {
    'train_acc': accuracy_score(y_train, y_train_pred_knn),
    'val_acc': accuracy_score(y_val, y_val_pred_knn),
    'test_acc': accuracy_score(y_test, y_test_pred_knn),
    'train_f1': f1_score(y_train, y_train_pred_knn),
    'val_f1': f1_score(y_val, y_val_pred_knn),
    'test_f1': f1_score(y_test, y_test_pred_knn),
}
print(f"   ✓ Train Acc: {resultados['kNN']['train_acc']:.4f}, Val Acc: {resultados['kNN']['val_acc']:.4f}")

# ===== MODELO 2: Decision Tree =====
print("\n2. Entrenando Decision Tree...")
dt = DecisionTreeClassifier(max_depth=10, random_state=42)
dt.fit(X_train_scaled, y_train)
modelos['Decision Tree'] = dt

y_train_pred_dt = dt.predict(X_train_scaled)
y_val_pred_dt = dt.predict(X_val_scaled)
y_test_pred_dt = dt.predict(X_test_scaled)

resultados['Decision Tree'] = {
    'train_acc': accuracy_score(y_train, y_train_pred_dt),
    'val_acc': accuracy_score(y_val, y_val_pred_dt),
    'test_acc': accuracy_score(y_test, y_test_pred_dt),
    'train_f1': f1_score(y_train, y_train_pred_dt),
    'val_f1': f1_score(y_val, y_val_pred_dt),
    'test_f1': f1_score(y_test, y_test_pred_dt),
}
print(f"   ✓ Train Acc: {resultados['Decision Tree']['train_acc']:.4f}, Val Acc: {resultados['Decision Tree']['val_acc']:.4f}")

# ===== MODELO 3: Random Forest =====
print("\n3. Entrenando Random Forest...")
rf = RandomForestClassifier(n_estimators=100, max_depth=15, random_state=42, n_jobs=-1)
rf.fit(X_train_scaled, y_train)
modelos['Random Forest'] = rf

y_train_pred_rf = rf.predict(X_train_scaled)
y_val_pred_rf = rf.predict(X_val_scaled)
y_test_pred_rf = rf.predict(X_test_scaled)

resultados['Random Forest'] = {
    'train_acc': accuracy_score(y_train, y_train_pred_rf),
    'val_acc': accuracy_score(y_val, y_val_pred_rf),
    'test_acc': accuracy_score(y_test, y_test_pred_rf),
    'train_f1': f1_score(y_train, y_train_pred_rf),
    'val_f1': f1_score(y_val, y_val_pred_rf),
    'test_f1': f1_score(y_test, y_test_pred_rf),
}
print(f"   ✓ Train Acc: {resultados['Random Forest']['train_acc']:.4f}, Val Acc: {resultados['Random Forest']['val_acc']:.4f}")

# ===== MODELO 4: Gradient Boosting =====
print("\n4. Entrenando Gradient Boosting...")
gb = GradientBoostingClassifier(n_estimators=100, max_depth=5, learning_rate=0.1, random_state=42)
gb.fit(X_train_scaled, y_train)
modelos['Gradient Boosting'] = gb

y_train_pred_gb = gb.predict(X_train_scaled)
y_val_pred_gb = gb.predict(X_val_scaled)
y_test_pred_gb = gb.predict(X_test_scaled)

resultados['Gradient Boosting'] = {
    'train_acc': accuracy_score(y_train, y_train_pred_gb),
    'val_acc': accuracy_score(y_val, y_val_pred_gb),
    'test_acc': accuracy_score(y_test, y_test_pred_gb),
    'train_f1': f1_score(y_train, y_train_pred_gb),
    'val_f1': f1_score(y_val, y_val_pred_gb),
    'test_f1': f1_score(y_test, y_test_pred_gb),
}
print(f"   ✓ Train Acc: {resultados['Gradient Boosting']['train_acc']:.4f}, Val Acc: {resultados['Gradient Boosting']['val_acc']:.4f}")

print(f"\n✓ Primeros 4 modelos entrenados!")

In [ ]:
# ===== MODELO 5: Deep Neural Network =====
print("\n5. Entrenando Deep Neural Network...")
print("   Arquitectura: 4 capas ocultas con Dropout para regularización")

dnn = keras.Sequential([
    layers.Dense(128, activation='relu', input_shape=(X_train_scaled.shape[1],)),
    layers.Dropout(0.3),
    
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.3),
    
    layers.Dense(32, activation='relu'),
    layers.Dropout(0.2),
    
    layers.Dense(16, activation='relu'),
    
    layers.Dense(1, activation='sigmoid')
])

dnn.compile(optimizer=Adam(learning_rate=0.001),
            loss='binary_crossentropy',
            metrics=['accuracy'])

early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

history = dnn.fit(
    X_train_scaled, y_train,
    validation_data=(X_val_scaled, y_val),
    epochs=100,
    batch_size=16,
    callbacks=[early_stop],
    verbose=0
)

y_train_pred_dnn = (dnn.predict(X_train_scaled, verbose=0) > 0.5).astype(int).flatten()
y_val_pred_dnn = (dnn.predict(X_val_scaled, verbose=0) > 0.5).astype(int).flatten()
y_test_pred_dnn = (dnn.predict(X_test_scaled, verbose=0) > 0.5).astype(int).flatten()

resultados['DNN'] = {
    'train_acc': accuracy_score(y_train, y_train_pred_dnn),
    'val_acc': accuracy_score(y_val, y_val_pred_dnn),
    'test_acc': accuracy_score(y_test, y_test_pred_dnn),
    'train_f1': f1_score(y_train, y_train_pred_dnn),
    'val_f1': f1_score(y_val, y_val_pred_dnn),
    'test_f1': f1_score(y_test, y_test_pred_dnn),
}
modelos['DNN'] = dnn

print(f"   ✓ Train Acc: {resultados['DNN']['train_acc']:.4f}, Val Acc: {resultados['DNN']['val_acc']:.4f}")

# Graficar curvas de entrenamiento del DNN
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history['loss'], label='Train Loss', linewidth=2)
plt.plot(history.history['val_loss'], label='Val Loss', linewidth=2)
plt.title('Pérdida (Loss) durante el Entrenamiento - DNN', fontweight='bold')
plt.xlabel('Época')
plt.ylabel('Loss')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(history.history['accuracy'], label='Train Accuracy', linewidth=2)
plt.plot(history.history['val_accuracy'], label='Val Accuracy', linewidth=2)
plt.title('Accuracy durante el Entrenamiento - DNN', fontweight='bold')
plt.xlabel('Época')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n✓ Todos los 5 modelos entrenados!")

In [ ]:
print("\n" + "="*70)
print("TABLA COMPARATIVA DE RESULTADOS")
print("="*70)

# Crear DataFrame con resultados
df_resultados = pd.DataFrame(resultados).round(4)

print("\n--- ACCURACY ---")
print(df_resultados.loc[['train_acc', 'val_acc', 'test_acc']])

print("\n--- F1-SCORE ---")
print(df_resultados.loc[['train_f1', 'val_f1', 'test_f1']])

# Identificar mejor modelo
best_model_name = df_resultados.loc['test_acc'].idxmax()
best_test_acc = df_resultados.loc['test_acc'].max()

print(f"\n{'='*70}")
print(f"MEJOR MODELO EN TEST: {best_model_name}")
print(f"Accuracy en Test: {best_test_acc:.4f}")
print(f"{'='*70}")

# Detectar overfitting
print("\n--- ANÁLISIS DE OVERFITTING ---")
print("(Diferencia entre Train Accuracy y Val Accuracy)")
for modelo_name in resultados.keys():
    diff = resultados[modelo_name]['train_acc'] - resultados[modelo_name]['val_acc']
    if diff > 0.10:
        status = "⚠️ OVERFITTING EVIDENTE"
    elif diff > 0.05:
        status = "✓ Ligero overfitting"
    else:
        status = "✓ Bien balanceado"
    print(f"  {modelo_name}: {diff:.4f} {status}")

## PARTE 6: Evaluación Final del Mejor Modelo

In [ ]:
print("\n" + "="*70)
print(f"EVALUACIÓN FINAL: {best_model_name}")
print("="*70)

best_model = modelos[best_model_name]

# Hacer predicciones finales en test
if best_model_name == 'DNN':
    y_test_pred_final = (best_model.predict(X_test_scaled, verbose=0) > 0.5).astype(int).flatten()
else:
    y_test_pred_final = best_model.predict(X_test_scaled)

# Calcular métricas finales
accuracy_final = accuracy_score(y_test, y_test_pred_final)
precision_final = precision_score(y_test, y_test_pred_final)
recall_final = recall_score(y_test, y_test_pred_final)
f1_final = f1_score(y_test, y_test_pred_final)

print(f"\n{'MÉTRICAS FINALES EN TEST SET':^70}")
print(f"{'─'*70}")
print(f"Accuracy:  {accuracy_final:.4f}  (% de predicciones correctas)")
print(f"Precision: {precision_final:.4f}  (% de 'Fatiga' predichas que son correctas)")
print(f"Recall:    {recall_final:.4f}   (% de casos reales de 'Fatiga' detectados)")
print(f"F1-Score:  {f1_final:.4f}   (balance entre Precision y Recall)")

# Matriz de Confusión
cm = confusion_matrix(y_test, y_test_pred_final)
print(f"\nMatriz de Confusión:")
print(f"                Predicción")
print(f"              Normal  Fatiga")
print(f"Real Normal  {cm[0,0]:5d}   {cm[0,1]:5d}")
print(f"     Fatiga  {cm[1,0]:5d}   {cm[1,1]:5d}")

# Visualizar matriz de confusión
fig, ax = plt.subplots(1, 2, figsize=(14, 5))

# Heatmap
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Normal', 'Fatiga'], 
            yticklabels=['Normal', 'Fatiga'],
            ax=ax[0], cbar_kws={'label': 'Cantidad'})
ax[0].set_title(f'Matriz de Confusión - {best_model_name}', fontweight='bold')
ax[0].set_ylabel('Real')
ax[0].set_xlabel('Predicción')

# Métricas
metrics_names = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
metrics_values = [accuracy_final, precision_final, recall_final, f1_final]
colors = ['#2E86AB', '#A23B72', '#F18F01', '#C73E1D']

ax[1].barh(metrics_names, metrics_values, color=colors)
ax[1].set_xlim([0, 1])
ax[1].set_xlabel('Score')
ax[1].set_title('Métricas de Desempeño', fontweight='bold')
ax[1].grid(True, alpha=0.3, axis='x')

for i, v in enumerate(metrics_values):
    ax[1].text(v + 0.02, i, f'{v:.4f}', va='center')

plt.tight_layout()
plt.show()

print("\n" + "-"*70)
print("REPORTE DE CLASIFICACIÓN DETALLADO:")
print("-"*70)
print(classification_report(y_test, y_test_pred_final, 
                          target_names=['Normal', 'Fatiga']))

# Interpretación
print("\n" + "="*70)
print("INTERPRETACIÓN DE RESULTADOS")
print("="*70)
if accuracy_final > 0.85:
    print("✓ EXCELENTE: El modelo tiene muy buen desempeño")
elif accuracy_final > 0.75:
    print("✓ BUENO: El modelo tiene desempeño aceptable")
else:
    print("⚠️ REGULAR: Se podría mejorar el modelo")

## PARTE 7: Prueba con Datos Artificiales

In [ ]:
print("\n" + "="*70)
print("PRUEBA CON MUESTRAS ARTIFICIALES")
print("="*70)

# Generar 2 muestras artificiales basadas en estadísticos reales

# Muestra 1: Características de condición NORMAL
muestra_normal = np.zeros(len(feature_columns))
for i, col in enumerate(feature_columns):
    stats_normal = df_features[df_features['Target'] == 0][col]
    muestra_normal[i] = np.random.normal(stats_normal.mean(), stats_normal.std())

# Muestra 2: Características con FATIGA
muestra_fatiga = np.zeros(len(feature_columns))
for i, col in enumerate(feature_columns):
    stats_fatiga = df_features[df_features['Target'] == 1][col]
    muestra_fatiga[i] = np.random.normal(stats_fatiga.mean(), stats_fatiga.std())

# Normalizar con el scaler entrenado
muestra_normal_scaled = scaler.transform(muestra_normal.reshape(1, -1))[0]
muestra_fatiga_scaled = scaler.transform(muestra_fatiga.reshape(1, -1))[0]

# Predicción
if best_model_name == 'DNN':
    pred_normal = best_model.predict(muestra_normal_scaled.reshape(1, -1), verbose=0)[0][0] > 0.5
    pred_fatiga = best_model.predict(muestra_fatiga_scaled.reshape(1, -1), verbose=0)[0][0] > 0.5
else:
    pred_normal = best_model.predict(muestra_normal_scaled.reshape(1, -1))[0]
    pred_fatiga = best_model.predict(muestra_fatiga_scaled.reshape(1, -1))[0]

print("\n" + "-"*70)
print("MUESTRA 1: Generada con VALORES DE CONDICIÓN NORMAL")
print("-"*70)
print(f"Características (primeras 10): {muestra_normal[:10].round(4)}")
print(f"Predicción del {best_model_name}: {'FATIGA' if pred_normal else 'NORMAL'} ({int(pred_normal)})")
print(f"¿Tiene sentido? {'✓ SÍ - Predice Normal como se esperaba' if not pred_normal else '✗ NO - Error inesperado'}")

print("\n" + "-"*70)
print("MUESTRA 2: Generada con VALORES DE FATIGA MUSCULAR")
print("-"*70)
print(f"Características (primeras 10): {muestra_fatiga[:10].round(4)}")
print(f"Predicción del {best_model_name}: {'FATIGA' if pred_fatiga else 'NORMAL'} ({int(pred_fatiga)})")
print(f"¿Tiene sentido? {'✓ SÍ - Predice Fatiga como se esperaba' if pred_fatiga else '✗ NO - Error inesperado'}")

print("\n" + "="*70)
print("CONCLUSIÓN")
print("="*70)
print(f"El modelo {best_model_name} fue capaz de clasificar correctamente")
print("muestras generadas artificialmente. Esto indica que el modelo")
print("ha aprendido patrones biológicos válidos sobre la fatiga muscular,")
print("y no solo ha memorizado los datos de entrenamiento.")

## CONCLUSIONES Y RECOMENDACIONES

### Respuestas a las Preguntas del Trabajo

**1. ¿Cuál modelo tuvo mejor desempeño?**
El modelo `{best_model_name}` tuvo el mejor desempeño con un Accuracy de {best_test_acc:.4f} en el conjunto de test.

**2. ¿Alguno presentó overfitting?**
Ver análisis anterior. El overfitting ocurre cuando Train Accuracy >> Val Accuracy. Los modelos con poca diferencia están bien balanceados.

**3. ¿Cuál seleccionaría para producción?**
Elegir el modelo con mejor balance entre:
- Accuracy alto en test
- Poca diferencia train-val (sin overfitting)
- Recall alto (detectar fatiga es crítico)
- Tiempo de predicción rápido

### Buen Trabajo
✓ Dataset correctamente preprocesado a formato binario
✓ Extracción de 7 características por músculo (56 totales)
✓ EDA completo con visualizaciones
✓ 5 modelos diferentes entrenados y comparados
✓ Matriz de confusión y métricas detalladas
✓ Prueba exitosa con datos artificiales

### Posibles Mejoras Futuras
- Usar más datos (tenemos 100,000 muestras disponibles)
- Ajuste fino de hiperparámetros (GridSearch)
- Feature selection para identificar características más importantes
- Balanceo de clases si hay desbalance
- Técnicas de ensamble (voting, stacking)